# Data preprocessing for the Twitter dataset

In [36]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
from langdetect import detect, DetectorFactory, LangDetectException
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import torch
from tqdm import tqdm
import nltk
nltk.download("punkt")
nltk.download("wordnet")

import pandas as pd
import re
import string
from langdetect import detect, LangDetectException
from tqdm.auto import tqdm
import spacy
from nltk.corpus import stopwords

import nltk
nltk.download('stopwords')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\malte\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [37]:
import pandas as pd
import os

# Verzeichnispfad
data_path = "raw"

# 1. CSV-Dateien einlesen
musk_quote_tweets = pd.read_csv(os.path.join(data_path, "musk_quote_tweets.csv"))
all_musk_posts = pd.read_csv(os.path.join(data_path, "all_musk_posts.csv"))

#shape
print(f"all_musk_posts shape: {all_musk_posts.shape}")

# 2. Neue Spalte 'quote_and_original' erzeugen
musk_quote_tweets["quote_and_original"] = (
    musk_quote_tweets["musk_quote_tweet"].astype(str) +
    " // " +
    musk_quote_tweets["orig_tweet_text"].astype(str)
)

# 3. Nur relevante Spalten für den Merge vorbereiten
merge_df = musk_quote_tweets[["musk_tweet_id", "quote_and_original"]]

# 4. Merge auf 'id' in all_musk_posts und 'musk_tweet_id' in merge_df
all_musk_posts_with_quotes = all_musk_posts.merge(
    merge_df,
    how="left",
    left_on="id",
    right_on="musk_tweet_id"
)

# 5. Spalte 'musk_tweet_id' entfernen
all_musk_posts_with_quotes.drop(columns=["musk_tweet_id"], inplace=True)

#shape
print(f"merged shape: {all_musk_posts_with_quotes.shape}")

# 6. Neue CSV speichern
output_path = os.path.join(data_path, "all_musk_posts_with_quotes.csv")
all_musk_posts_with_quotes.to_csv(output_path, index=False)

print("✅ CSV erfolgreich gespeichert unter:", output_path)


C:\Users\malte\AppData\Local\Temp\ipykernel_26520\3812318178.py:9: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  all_musk_posts = pd.read_csv(os.path.join(data_path, "all_musk_posts.csv"))


all_musk_posts shape: (55099, 24)
merged shape: (55099, 25)
✅ CSV erfolgreich gespeichert unter: raw\all_musk_posts_with_quotes.csv


In [38]:
musk_twitter_data = pd.read_csv(os.path.join('raw', 'all_musk_posts_with_quotes.csv'),parse_dates=["createdAt"])

start_date = "2015-01-01"
end_date = musk_twitter_data["createdAt"].max()

# end_date = musk_twitter_data["createdAt"].max()

musk_twitter_data = musk_twitter_data[musk_twitter_data["createdAt"] > start_date]
musk_twitter_data = musk_twitter_data[musk_twitter_data["createdAt"] < end_date]

musk_twitter_data["isRetweet"] = musk_twitter_data["isRetweet"].astype(str).str.lower()
musk_twitter_data["possiblySensitive"] = musk_twitter_data["possiblySensitive"].astype(str).str.lower()
musk_twitter_data["fullText"] = musk_twitter_data["fullText"].astype(str)
musk_twitter_data["date"] = musk_twitter_data["createdAt"].dt.date

#shape ausgeben
print(f"Shape of musk_twitter_data: {musk_twitter_data.shape}")

Shape of musk_twitter_data: (16808, 26)


C:\Users\malte\AppData\Local\Temp\ipykernel_26520\3932034708.py:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data = pd.read_csv(os.path.join('raw', 'all_musk_posts_with_quotes.csv'),parse_dates=["createdAt"])


In [39]:
#EMOJI encoding

import os
import emoji
import pandas as pd

# ⚠️ Annahme: musk_twitter_data ist bereits im Arbeitsspeicher geladen

# 🔧 Relevante Spalten definieren
text_columns = ["fullText", "quote_and_original"]

# 🔄 Emojis in Kurztextform bringen (z. B. "🚀" → ":rocket:")
for col in text_columns:
    musk_twitter_data[col] = musk_twitter_data[col].astype(str).apply(
        lambda txt: emoji.demojize(txt, language="en")
    )

# 💾 Als neue CSV speichern
output_dir = "raw"
output_path = os.path.join(output_dir, "musk_twitter_data_demojized.csv")
musk_twitter_data.to_csv(output_path, index=False, encoding="utf-8")

print(f"✅ Demojized CSV gespeichert unter: {output_path}")


✅ Demojized CSV gespeichert unter: raw\musk_twitter_data_demojized.csv


In [40]:
# --- Engagement Index für späteres Weighting in "2. Feature_engineering.ipynb" berechnen ---
import numpy as np

# 1. Spalten definieren
count_cols = ['retweetCount', 'replyCount', 'likeCount', 'quoteCount']

# 2. Min/Max je Spalte
mins = musk_twitter_data[count_cols].min()
maxs = musk_twitter_data[count_cols].max()

# 3. Gewichtungen (erstmal 1/4, können später angepasst werden)
weights = {
    'retweetCount': 0.25,
    'replyCount':    0.25,
    'likeCount':     0.25,
    'quoteCount':    0.25,
}
weights_series = pd.Series(weights)

# 4. Min–Max-Normalisierung (ohne Log)
normed = (musk_twitter_data[count_cols] - mins) / (maxs - mins)

# 5. Numerator & Denominator für gewichteten Durchschnitt (fehlende ignorieren)
weighted_normed = normed.mul(weights_series, axis=1)
denominator = normed.notna().mul(weights_series, axis=1).sum(axis=1)
numerator   = weighted_normed.sum(axis=1)

# 6. Engagement-Index berechnen und Inf→NaN ersetzen
ei = numerator / denominator
ei = ei.replace([np.inf, -np.inf], np.nan)

# 7. ins DataFrame schreiben
musk_twitter_data['engagement_index'] = ei
# --- Ende Engagement Index ---

In [41]:
#TODO @malte: missing values bei engagement index und den vier metrics irgendwie handeln

In [42]:
#auftrennen in ..._all und ..._nlp
musk_twitter_data_all = musk_twitter_data.copy()
musk_twitter_data_nlp = musk_twitter_data.copy()

#shape ausgeben
musk_twitter_data_nlp.shape, musk_twitter_data_all.shape, musk_twitter_data.shape

((16808, 27), (16808, 27), (16808, 27))

In [43]:
# # --- Setup ---
# tqdm.pandas()
# nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
# stop_words = set(stopwords.words('english'))
# 
# # --- Helpers ---
# def safe_detect(text):
#     try:
#         text = str(text).strip()
#         if len(text) < 10:
#             return "unknown"
#         return detect(text)
#     except LangDetectException:
#         return "unknown"
# 
# def clean_basic(text):
#     text = str(text)
#     text = re.sub(r"http\S+|www\S+|@\w+|#|RT", "", text)
#     return text.strip()
# 
# def clean_for_topic(text):
#     text = clean_basic(text).lower()
#     text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
#     text = re.sub(r"\d+", "", text)
#     return text
# 
# def preprocess_lemmatized(text):
#     text = clean_for_topic(text)
#     doc = nlp(text)
#     return " ".join([token.lemma_ for token in doc if token.is_alpha and token.lemma_ not in stop_words])
# 
# # --- NLP DataFrame creation ---
# musk_twitter_data_nlp = musk_twitter_data.copy()
# 
# # Filter for valid original texts
# musk_twitter_data_nlp = musk_twitter_data_nlp[
#     (musk_twitter_data_nlp["isRetweet"] != "true") &
#     (musk_twitter_data_nlp["fullText"].str.strip() != "")
# ]
# 
# # Language detection
# musk_twitter_data_nlp["language"] = musk_twitter_data_nlp["fullText"].progress_apply(safe_detect)
# musk_twitter_data_nlp = musk_twitter_data_nlp[musk_twitter_data_nlp["language"] == "en"]
# 
# # Text variations
# musk_twitter_data_nlp["text_raw"] = musk_twitter_data_nlp["fullText"].progress_apply(clean_basic)
# musk_twitter_data_nlp["text_lemmatized"] = musk_twitter_data_nlp["fullText"].progress_apply(preprocess_lemmatized)

In [44]:
# import pandas as pd
# import re, string
# from tqdm import tqdm
# import spacy
# from langdetect import detect, LangDetectException
# from nltk.corpus import stopwords
# 
# # Initialisierung
# tqdm.pandas()
# nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
# stop_words = set(stopwords.words('english'))
# 
# # Hilfsfunktionen
# def safe_detect(text):
#     try:
#         text = str(text).strip()
#         if len(text) < 10:
#             return "unknown"
#         return detect(text)
#     except LangDetectException:
#         return "unknown"
# 
# def clean_basic(text):
#     text = str(text)
#     text = re.sub(r"http\S+|www\S+|@\w+|#|RT", "", text)
#     return text.strip()
# 
# def clean_for_topic(text):
#     text = clean_basic(text).lower()
#     text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
#     text = re.sub(r"\d+", "", text)
#     return text
# 
# def preprocess_lemmatized(text):
#     text = clean_for_topic(text)
#     doc = nlp(text)
#     return " ".join([token.lemma_ for token in doc if token.is_alpha and token.lemma_ not in stop_words])
# 
# # Initialleerer Textfeld-Spalte
# musk_twitter_data_nlp["text_raw"] = ""
# musk_twitter_data_nlp["text_lemmatized"] = ""
# musk_twitter_data_nlp["quote_and_original_raw"] = ""
# musk_twitter_data_nlp["quote_and_original_lemmatized"] = ""
# musk_twitter_data_nlp["language"] = ""
# 
# 
# # Filter für Standardtweets (keine Quote-Tweets)
# nonquote_mask = (
#     (musk_twitter_data_nlp["isQuote"] != True) &
#     (musk_twitter_data_nlp["isRetweet"] != "true") &
#     (musk_twitter_data_nlp["fullText"].str.strip().str.len() >= 10)
# )
# 
# # Sprache erkennen und filtern
# musk_twitter_data_nlp.loc[nonquote_mask, "language"] = musk_twitter_data_nlp.loc[nonquote_mask, "fullText"].progress_apply(safe_detect)
# musk_twitter_data_nlp = musk_twitter_data_nlp[
#     ~nonquote_mask | (musk_twitter_data_nlp["language"] == "en")
# ]
# 
# 
# 
# # Iteration Zeile für Zeile
# for idx, row in tqdm(musk_twitter_data_nlp.iterrows(), total=len(musk_twitter_data_nlp)):
# 
#     is_quote = row.get("isQuote", False)
#     quote_text = row.get("quote_and_original", None)
#     base_text = str(row.get("fullText", "")).strip()
# 
#     # Standardfall: kein Quote-Tweet oder ungültiges quote_and_original
#     if not is_quote or pd.isna(quote_text) or str(quote_text).strip() == "":
#         lang = safe_detect(base_text)
#         musk_twitter_data_nlp.at[idx, "language"] = lang
# 
#         if lang == "en":
#             musk_twitter_data_nlp.at[idx, "text_raw"] = clean_basic(base_text)
#             musk_twitter_data_nlp.at[idx, "text_lemmatized"] = preprocess_lemmatized(base_text)
# 
#     # Spezialfall: Quote-Tweet mit quote_and_original vorhanden
#     else:
#         # Sprache wird hier gespeichert, aber **nicht** gefiltert
#         musk_twitter_data_nlp.at[idx, "language"] = safe_detect(quote_text)
#         musk_twitter_data_nlp.at[idx, "quote_and_original_raw"] = clean_basic(quote_text)
#         musk_twitter_data_nlp.at[idx, "quote_and_original_lemmatized"] = preprocess_lemmatized(quote_text)
# 
# 
# musk_twitter_data_nlp.to_csv("raw/all_musk_posts_with_quotes_nlp_TEMPORARY.csv", index=False)
# print("✅ NLP-verarbeitete Datei gespeichert.")


In [45]:
import pandas as pd
import re, string
tqdm.pandas()

# NLP-Imports
import spacy
from langdetect import detect, LangDetectException
from nltk.corpus import stopwords

# Initialisierung
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
stop_words = set(stopwords.words('english'))

# Hilfsfunktionen
def safe_detect(text):
    try:
        text = str(text).strip()
        if len(text) < 10:
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"

def clean_basic(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+|@\w+|#|RT", "", text)
    return text.strip()

def clean_for_topic(text):
    text = clean_basic(text).lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\d+", "", text)
    return text

def preprocess_lemmatized(text):
    text = clean_for_topic(text)
    doc = nlp(text)
    return " ".join([
        token.lemma_ for token in doc
        if token.is_alpha and token.lemma_ not in stop_words
    ])

# Spalten initialisieren
cols = [
    'text_raw', 'text_lemmatized',
    'quote_and_original_raw', 'quote_and_original_lemmatized',
    'language'
]
musk_twitter_data_nlp = musk_twitter_data_nlp.assign(**{c: '' for c in cols})

# 1) Sprache nur für non-Quote, non-Retweet ermitteln
mask_nonquote = (
    (~musk_twitter_data_nlp['isQuote'])
    & (musk_twitter_data_nlp['isRetweet'] != 'true')
)
musk_twitter_data_nlp.loc[mask_nonquote, 'language'] = (
    musk_twitter_data_nlp.loc[mask_nonquote, 'fullText']
    .progress_apply(safe_detect)
)

# 2) DataFrame filtern: 
mask_keep = (
    # alle Quote-Tweets behalten
    musk_twitter_data_nlp['isQuote']
    |
    # non-Quote & non-Retweet: Länge>=10 & Sprache==en
    (
        ~musk_twitter_data_nlp['isQuote']
        & (musk_twitter_data_nlp['isRetweet'] != 'true')
        & (musk_twitter_data_nlp['fullText'].str.len() >= 10)
        & (musk_twitter_data_nlp['language'] == 'en')
    )
)
musk_twitter_data_nlp = musk_twitter_data_nlp[mask_keep].copy()

# 3) Loop zur Befüllung der Text-Spalten
for idx, row in tqdm(musk_twitter_data_nlp.iterrows(), total=len(musk_twitter_data_nlp)):
    if row['isQuote']:
        quote_text = row.get('quote_and_original', '') or ''
        musk_twitter_data_nlp.at[idx, 'quote_and_original_raw'] = clean_basic(quote_text)
        musk_twitter_data_nlp.at[idx, 'quote_and_original_lemmatized'] = preprocess_lemmatized(quote_text)
    else:
        txt = row.get('fullText', '') or ''
        musk_twitter_data_nlp.at[idx, 'text_raw'] = clean_basic(txt)
        musk_twitter_data_nlp.at[idx, 'text_lemmatized'] = preprocess_lemmatized(txt)

print("✅ NLP-verarbeitete Datei gespeichert.")


  0%|          | 0/10835 [00:00<?, ?it/s]

  0%|          | 0/14039 [00:00<?, ?it/s]

✅ NLP-verarbeitete Datei gespeichert.


In [46]:
print("Number of tweets after filtering: ", musk_twitter_data_nlp.shape[0])
print("Number of tweets before filtering: ", musk_twitter_data.shape[0])

display(musk_twitter_data_nlp.head()) 


Number of tweets after filtering:  14039
Number of tweets before filtering:  16808


,id,url,twitterUrl,fullText,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,...,quote,retweet,quote_and_original,date,engagement_index,text_raw,text_lemmatized,quote_and_original_raw,quote_and_original_lemmatized,language
6910,1881123459046260863,https://x.com/elonmusk/status/1881123459046260863,https://twitter.com/elonmusk/status/1881123459...,@RealAlexJones America is the central pillar o...,674.0,860.0,7776.0,89.0,316075.0,2025-01-19 23:35:56+00:00,...,NaN,NaN,nan,2025-01-19,0.003861,America is the central pillar of civilization,america central pillar civilization,,,en
6911,1881113728948764887,https://x.com/elonmusk/status/1881113728948764887,https://twitter.com/elonmusk/status/1881113728...,"@ElonClipsX Wow, thanks Piers",115.0,215.0,1785.0,11.0,48382.0,2025-01-19 22:57:17+00:00,...,NaN,NaN,nan,2025-01-19,0.000803,"Wow, thanks Piers",wow thank pier,,,en
6912,1881097930570104983,https://x.com/elonmusk/status/1881097930570104983,https://twitter.com/elonmusk/status/1881097930...,@sully40272 @ScottAdamsSays Yeah,47.0,66.0,618.0,7.0,38715.0,2025-01-19 21:54:30+00:00,...,NaN,NaN,nan,2025-01-19,0.000255,Yeah,yeah,,,en
6913,1881097521910698333,https://x.com/elonmusk/status/1881097521910698333,https://twitter.com/elonmusk/status/1881097521...,@pranjalssh Yes. Aiming for truth with least e...,64.0,83.0,955.0,7.0,104394.0,2025-01-19 21:52:52+00:00,...,NaN,NaN,nan,2025-01-19,0.000340,Yes. Aiming for truth with least error is inhe...,yes aim truth least error inherently base,,,en
6915,1881091979427381342,https://x.com/elonmusk/status/1881091979427381342,https://twitter.com/elonmusk/status/1881091979...,@Rothmus LotR has no equal,125.0,240.0,2001.0,33.0,142643.0,2025-01-19 21:30:51+00:00,...,NaN,NaN,nan,2025-01-19,0.001042,LotR has no equal,lotr equal,,,en


In [47]:
# To csv
musk_twitter_data_nlp.to_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'), index=False)
musk_twitter_data_all.to_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'), index=False)